# MCP + Generative UI 에이전트 (스트리밍)

```
사용자 질문
  → 에이전트가 MCP 도구 호출로 데이터 획득
  → 도구 결과의 "형태"를 코드로 판별
      list[dict] (표 형태)      → type: "data-table"   + props  ← A 응답
      dict  (단일 레코드)       → type: "weather-card" + props  ← B 응답
  → 클라이언트: 요약 텍스트는 토큰 스트리밍으로, UI 페이로드는 구조화 이벤트로 수신
```

설계 포인트 두 가지:

1. **형태 판별은 LLM이 아니라 코드가 한다.** LLM에게 "표면 A형식으로 답해" 라고 시키면 소형 모델은
   자주 어긴다. 도구 결과의 타입/스키마 검사는 결정론적 로직으로 처리하고, LLM은 요약 텍스트만 담당.
2. **스트림은 이벤트 채널이다.** `token`(텍스트 조각)과 `ui`(type+props) 두 종류 이벤트를 한 스트림에
   흘리고, 클라이언트가 이벤트 종류별로 처리한다 — SSE 로 그대로 옮겨지는 계약.

> 사전 조건: Ollama + `gemma4:e4b`. MCP 서버(`../mcp_server/server.py`)는 노트북이 stdio 서브프로세스로 자동 실행한다.

## 1. MCP 도구 로드

`langchain-mcp-adapters` 가 MCP 서버의 도구를 LangChain tool 로 변환해준다.
서버는 stdio 로 뜨므로 별도 실행 불필요 — 클라이언트가 서브프로세스로 띄운다.

(Jupyter 는 top-level `await` 를 지원한다)

In [1]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "demo-data": {
            "url": "http://localhost:8000/mcp",
            "transport": "streamable_http",
        }
    }
)

tools = await mcp_client.get_tools()
for t in tools:
    print(f"{t.name}: {t.description}")

search_books: 도서 목록을 검색한다. genre 를 주면 해당 장르만, 없으면 전체를 반환한다.
get_weather: 도시의 현재 날씨를 조회한다. 지원 도시: 서울, 부산, 제주


## 2. 에이전트 구성

In [2]:
from langchain_ollama import ChatOllama

SYSTEM = (
    "너는 데이터 조회 어시스턴트다. 질문에 답하려면 도구를 사용해 데이터를 얻어라. "
    "데이터를 얻은 뒤에는 핵심을 2~3문장으로 요약해서 답하라. "
    "표나 수치를 텍스트로 길게 나열하지 마라 — 상세 데이터는 UI가 따로 표시한다."
)

llm = ChatOllama(model="gemma4:e4b", base_url="http://localhost:11434")

try:  # langchain v1
    from langchain.agents import create_agent
    agent = create_agent(llm, tools=tools, system_prompt=SYSTEM)
except ImportError:
    from langgraph.prebuilt import create_react_agent
    agent = create_react_agent(llm, tools=tools, prompt=SYSTEM)

## 3. 형태 판별 → UI 페이로드 (A / B 분기)

MCP 도구 결과는 JSON 문자열로 온다. 파싱해서:

- `list[dict]` → **A: data-table** — 클라이언트가 그리드 컴포넌트에 꽂을 수 있는 columns/rows
- 날씨 스키마의 `dict` → **B: weather-card** — 카드 컴포넌트 props
- 그 외 → text (폴백)

실서비스에서는 이 함수가 그래프의 마지막 노드가 된다.

In [3]:
import json


def parse_tool_output(raw) -> object:
    """MCP 결과를 벗긴다: JSON 문자열 → 객체, content block 리스트 → 내용물"""
    if isinstance(raw, str):
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            return raw
    if isinstance(raw, list):
        items = []
        for x in raw:
            if isinstance(x, dict) and x.get("type") == "text":   # content block 껍질 벗기기
                items.append(parse_tool_output(x["text"]))
            else:
                items.append(parse_tool_output(x) if isinstance(x, str) else x)
        return items[0] if len(items) == 1 else items   # 블록 1개면 그대로 (날씨 dict)
    return raw


def build_ui_payload(data) -> dict:
    """데이터 형태 → {type, props} 분기 (결정론적)"""
    # list[dict] 를 JSON 문자열 리스트로 받는 경우까지 정규화
    if isinstance(data, list):
        items = [parse_tool_output(x) for x in data]
        if items and all(isinstance(x, dict) for x in items):
            return {
                "type": "data-table",                      # ← A 응답
                "props": {"columns": list(items[0].keys()), "rows": items},
            }
    if isinstance(data, dict) and {"city", "temp_c"} <= data.keys():
        return {"type": "weather-card", "props": data}     # ← B 응답
    return {"type": "text", "props": {"raw": str(data)[:500]}}  # 폴백


# 단위 테스트
print(build_ui_payload([{"a": 1}, {"a": 2}])["type"])                    # data-table
print(build_ui_payload({"city": "서울", "temp_c": 29})["type"])          # weather-card
print(build_ui_payload("hello")["type"])                                  # text

data-table
weather-card
text


## 4. 스트리밍 실행

`astream_events` 로 실행 이벤트를 실시간 수신한다:

- `on_chat_model_stream` → LLM 토큰 조각 → **token 이벤트**로 즉시 방출
- `on_tool_end` → 도구 결과 확보 (아직 방출 안 함)
- 스트림 종료 → 형태 판별 → **ui 이벤트** 방출

여기서는 클라이언트 역할을 print 로 흉내 낸다.

In [4]:
async def run_streaming(question: str):
    tool_result = None

    async for event in agent.astream_events({"messages": [("user", question)]}, version="v2"):
        kind = event["event"]
        if kind == "on_chat_model_stream":
            chunk = event["data"]["chunk"].content
            if chunk:
                print(chunk, end="", flush=True)          # → event: token
        elif kind == "on_tool_start":
            print(f"\n>> tool call: {event['name']}({event['data'].get('input')})")
        elif kind == "on_tool_end":
            output = event["data"]["output"]
            tool_result = parse_tool_output(getattr(output, "content", output))

    ui = build_ui_payload(tool_result) if tool_result is not None else None
    print("\n\n>> event: ui")                              # → event: ui
    print(json.dumps(ui, ensure_ascii=False, indent=2))


await run_streaming("판타지 장르 책 목록 보여줘")   # → A: data-table


>> tool call: search_books({'genre': '판타지'})
요청하신 판타지 장르의 책 목록은 '달빛 도서관'과 '시간의 정원사' 두 권이 있습니다. 이 책들은 각각 작가 김하늘(2021년)과 이서준(2019년)의 작품으로, 높은 평점(4.6점 및 4.3점)을 기록하며 인기를 얻고 있습니다.

>> event: ui
{
  "type": "data-table",
  "props": {
    "columns": [
      "title",
      "author",
      "genre",
      "year",
      "rating"
    ],
    "rows": [
      {
        "title": "달빛 도서관",
        "author": "김하늘",
        "genre": "판타지",
        "year": 2021,
        "rating": 4.6
      },
      {
        "title": "시간의 정원사",
        "author": "이서준",
        "genre": "판타지",
        "year": 2019,
        "rating": 4.3
      }
    ]
  }
}


In [5]:
await run_streaming("서울 날씨 어때?")   # → B: weather-card


>> tool call: get_weather({'city': '서울'})
서울의 현재 날씨는 맑으며 기온은 29도입니다. 습도는 62%, 바람은 시속 8km로 느껴지니 외출 시 참고하세요.

>> event: ui
{
  "type": "weather-card",
  "props": {
    "city": "서울",
    "temp_c": 29,
    "condition": "맑음",
    "humidity": 62,
    "wind_kmh": 8
  }
}


## 5. 실제 서비스로 옮기면 (계약 정리)

노트북의 print 두 종류가 곧 SSE 이벤트 계약이다. FastAPI 로 옮기면:

```
HTTP GET /chat?q=...   (Accept: text/event-stream)

event: token\ndata: {"text": "판타지 장르는 "}\n\n
event: token\ndata: {"text": "2권이 검색되었고..."}\n\n
event: ui\ndata: {"type": "data-table", "props": {"columns": [...], "rows": [...]}}\n\n
event: done\ndata: {}\n\n
```

프론트엔드는 type → 컴포넌트 매핑 테이블 하나로 끝난다:

```ts
const registry = {
  "data-table":   DataTable,     // props: { columns, rows }
  "weather-card": WeatherCard,   // props: { city, temp_c, condition, ... }
  "text":         PlainText,
};
// <component :is="registry[msg.type]" v-bind="msg.props" />
```

새 UI 타입 추가 = `build_ui_payload` 분기 하나 + 프론트 registry 항목 하나.
LLM 프롬프트는 건드릴 필요가 없다 — 형태 판별을 코드에 둔 덕분이다.